# CAM · Curso de estrategia
Ejecuta la siguiente celda e introduce la contraseña facilitada en el curso. No escribas la contraseña en una celda de código.

Después ejecuta los ejercicios en orden. El código editable y los notebooks se preparan automáticamente en **Archivos → curso-motorsport**. Haz doble clic en un archivo `.py` para leerlo y editarlo dentro de Colab. No necesitas descargar ni subir nada manualmente. Guarda una copia del notebook en Drive; los cambios en los archivos Python de la sesión son temporales.


In [ ]:
%pip install -q git+https://github.com/dcrespons14/cam-estrategia-aula.git
from cam_estrategia import besbloquear_cuaderno
besbloquear_cuaderno()


# Monte Carlo: cuándo activar el Attack Mode
Comparamos estrategias de un piloto frente a activaciones aleatorias de sus rivales. Cada estrategia se evalúa con los mismos escenarios rivales. La parrilla y las reglas de carrera se mantienen constantes.

Este primer ejemplo usa una activación. Incluimos las vueltas 3 a 25, como permite el simulador: el beneficio empieza en la vuelta siguiente y puede quedar duración sin aprovechar al final.

In [ ]:
from race_models.fe.config import RaceConfig
from race_models.fe.monte_carlo import run_monte_carlo
from race_models.fe.analysis.mc_plots import plot_monte_carlo

race_config = RaceConfig()

starting_position = 1  # 1 = pole; selecciona la posicion de salida.
simulations = 1000  # Carreras por estrategia.

activation_plans = [(lap,) for lap in range(3, race_config.total_laps + 1)]

In [ ]:
result = run_monte_carlo(
    race_config=race_config,
    starting_position=starting_position,
    activation_plans=activation_plans,
    simulations=simulations,
)
print(f"Carreras simuladas: {len(result.strategies) * len(result.scenario_seeds):,}")

## Comparar resultados
Priorizamos los **puntos medios**, seguidos de su STD, la posición media y su STD. Las tablas se ordenan de mayor a menor puntuación media; en caso de empate, por mejor posición media. Los gráficos muestran primero los puntos.

Los resultados se muestran como DataFrames (`df` y `df_sc`), con tres decimales y conservando los valores numéricos originales. STD es la desviación estándar muestral, no la varianza ni el error de la media. `NaN` indica muestra insuficiente para calcularla. Un plan vacío `()` indica que no se activó.

**Penalización:** un coche que acaba sin ninguna activación de Attack Mode pasa detrás de todos los que sí activaron. Si hay varios, conservan su orden relativo. Las posiciones y puntos usan esta clasificación final (`race_result.final_order`); el historial por vuelta conserva el orden en pista. Con varias activaciones configuradas, esta regla penaliza cero activaciones, no un plan parcialmente completado.


In [ ]:
import pandas as pd
from IPython.display import display
from race_models.fe.analysis.mc_tables import monte_carlo_dataframe

df = monte_carlo_dataframe(result)
with pd.option_context('display.max_rows', None, 'display.float_format', '{:.3f}'.format):
    display(df)

In [ ]:
df.to_csv("resultados_monte_carlo.csv", index=False)
print("Tabla guardada en resultados_monte_carlo.csv; descárgala desde Archivos.")

In [ ]:
plot_monte_carlo(result);

Las barras representan la incertidumbre de la media estimada, no la dispersión de las posiciones de una carrera individual. Son intervalos aproximados por estrategia; no prueban por sí solos que una estrategia sea superior a todas las demás. Aumenta `simulations` para estudiar la estabilidad de las estimaciones.

Los resultados individuales están en `strategy.final_positions`, en el mismo orden de escenarios para todas las estrategias. Para comparar varias activaciones, configura `RaceConfig(activations=2)` y proporciona planes válidos, por ejemplo `[(3, 10), (6, 15)]`.

## Ver una carrera individual
Los indices empiezan en cero. `strategy_index` sigue el orden de `activation_plans`, no el de la tabla ordenada. El mismo `race_index` utiliza los mismos rivales en todas las estrategias. La carrera se reconstruye con la semilla guardada, incluso si ejecutaste la Monte Carlo con `seed=None`.

In [ ]:
from race_models.fe.analysis.race_plots import plot_race

race_index = 0
activation_lap = 15

race_result = result.get_race(race_index, strategy_index=activation_lap - 3)
print("Activaciones:", result.strategies[activation_lap - 3].activation_laps)
plot_race(race_result, title=f"Monte Carlo - activación L{activation_lap}, carrera {race_index}");

## Monte Carlo con Safety Car
Activamos `sc=True`; con `sc=False` se mantiene el experimento anterior. En cada vuelta verde se sortea el inicio de un SC: 8 % en vueltas 1–5, 2 % en 6–20 y 8% desde la 21. Dura tres vueltas incluida la de inicio; no se sortean nuevos SC mientras hay uno activo. Puede haber varios SC en la carrera.

Bajo SC no se activa ni se adelanta, pero el Attack Mode ya activo sigue consumiendo su duración. Las activaciones pendientes pasan a la primera vuelta verde disponible; si todavía sigue activo un AM anterior, esperan a que termine. Si la carrera acaba antes, quedan sin ejecutar. No se añaden vueltas.

Se simulan las mismas carreras por estrategia prevista y los mismos escenarios de rivales y SC para todas. Guardamos las vueltas reales: una activación prevista en la 20 y ejecutada en la 21 cuenta como 21 en la agrupación inferior. Por eso cada grupo real puede contener un número diferente de carreras. Son medias descriptivas de escenarios mezclados, no una comparación causal entre estrategias; no mostramos intervalos de confianza en esa agrupación. Un plan vacío indica que no se llegó a activar.

In [ ]:
result_sc = run_monte_carlo(
    race_config=race_config,
    starting_position=starting_position,
    activation_plans=activation_plans,
    simulations=simulations,
    seed=None,
    sc=True,
)

In [ ]:
df_sc = monte_carlo_dataframe(result_sc, by_actual=True)
with pd.option_context('display.max_rows', None, 'display.float_format', '{:.3f}'.format):
    display(df_sc)

In [ ]:
plot_monte_carlo(result_sc, by_actual=True);

### Inspeccionar una carrera con SC
Los índices siguen refiriéndose a las estrategias previstas originales. El gráfico marca en amarillo las vueltas bajo SC y muestra las activaciones reales.

In [ ]:
race_index_sc = 4
strategy_index_sc = 4
race_sc = result_sc.get_race(race_index_sc, strategy_index_sc)
print('Vueltas bajo SC:', race_sc.sc_laps)
print('Plan previsto:', result_sc.strategies[strategy_index_sc].activation_laps)
print('Plan real:', result_sc.strategies[strategy_index_sc].actual_activation_laps[race_index_sc])
plot_race(race_sc, title=f'SC - estrategia {strategy_index_sc}, carrera {race_index_sc}');

## Comparar distribuciones de varias activaciones
Elige una lista de vueltas, por ejemplo `[15, 16, 17]`. Con `kde=False` se superponen histogramas y con `kde=True`, estimaciones de densidad con kernels gaussianos. Las selecciones repetidas se dibujan una sola vez. Cada vuelta mantiene su color en puntos y posiciones, y la leyenda muestra su número de carreras.

Cada histograma se normaliza al 100 % por separado. En KDE, el eje vertical representa **densidad**, no porcentaje; cada curva tiene área aproximada 1. La KDE es un suavizado de resultados discretos: puede asignar densidad a valores intermedios o fuera del rango posible, que no son resultados reales. Usa `bandwidth=1.0` para ajustar el suavizado en unidades del eje, o `None` para el ancho automático de Scott con un mínimo de 0.5. Este mínimo permite dibujar grupos constantes o de una sola carrera, sin implicar variabilidad observada.

Con `by_actual=False` comparas estrategias previstas. Con `by_actual=True` comparas vueltas reales, que agrupan escenarios diferentes y pueden tener distinto tamaño. Los resultados incluyen penalizaciones. Un grupo inexistente genera un error.

Una tupla indica un plan con varias activaciones: `[(3, 10), (6, 15)]` compara dos planes. `()` con `by_actual=True` selecciona el grupo sin activar.

In [ ]:
from race_models.fe.analysis.mc_plots import plot_monte_carlo_distribution

activation_laps_distribution = [22, 23]
use_kde = False  # False para histogramas superpuestos.
bandwidth = None  # Por ejemplo 1.0 para fijar el suavizado.
plot_monte_carlo_distribution(
    result, activation_lap=activation_laps_distribution,
    kde=use_kde, bandwidth=bandwidth, compact=False,
);

In [ ]:
by_actual_distribution = True  # False para comparar estrategias previstas.
plot_monte_carlo_distribution(
    result_sc, activation_lap=activation_laps_distribution,
    by_actual=by_actual_distribution, kde=use_kde, bandwidth=bandwidth, compact=True,
);

Con `compact=True`, la figura mide 15.7 × 14.2 cm, con más espacio vertical para ambas distribuciones. El eje de puntos va de 25 a la izquierda a 0 a la derecha. Usa `compact=False` para el formato grande. Puedes volver a ejecutar solo la celda del gráfico: se conservan los mismos resultados simulados.